In [ ]:
import requests
import torch
from PIL import Image
from transformers import GroundingDino2ForObjectDetection, GroundingDino2Config, GroundingDino2Processor, GroundingDinoImageProcessor
from transformers import AutoTokenizer, CLIPImageProcessor, AutoProcessor
from typing import Tuple
from torch import Tensor
import matplotlib.pyplot as plt
from datasets import load_dataset
import albumentations as A
from typing import Any, Dict, List, Mapping, Tuple, Union
from transformers.image_processing_utils import BatchFeature
from functools import partial
import numpy as np
import random
import torchvision.transforms as transforms
from torch import tensor

## Understanding basic configuration

In [ ]:
config = GroundingDino2Config()
print(config.multimodal_config.vision_config)
print(config.multimodal_config.text_config)

In [ ]:
model_id = "IDEA-Research/grounding-dino-tiny"
device = "cuda"

image_processor = GroundingDinoImageProcessor()
tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch32")
semantic_processor = CLIPImageProcessor()

processor = GroundingDino2Processor(image_processor, tokenizer, semantic_processor)

In [ ]:
# Load image for inference
url = "https://images.pexels.com/photos/8413299/pexels-photo-8413299.jpeg?auto=compress&cs=tinysrgb&w=630&h=375&dpr=2"
image = Image.open(requests.get(url, stream=True).raw)
raw_image = image.copy()
image

In [ ]:
def crop_bboxes(image, bboxes):
    """
    Crop bounding boxes from a PIL image.

    Args:
        image (PIL.Image.Image): The input PIL image from which to crop the bounding boxes.
        bboxes (list of tuples): List of bounding boxes, where each bbox is represented as a tuple (x, y, w, h).

    Returns:
        list of PIL.Image.Image: List of cropped images corresponding to each bounding box.
    """
    cropped_images = []
    for (x, y, w, h) in bboxes:
        cropped_img = image[int(y):int(y+h),int(x):int(x+w)]
        cropped_images.append(cropped_img)
    
    return cropped_images

cropped_images = crop_bboxes(np.array(image.convert("RGB")), [(420,370,190,170),(325,150,200,150)])


In [ ]:
for cropped_image in cropped_images:
    plt.imshow(cropped_image)
    plt.show()

In [ ]:
text = ["Mask. Cover. Ultralytics."]
inputs = processor(images=raw_image, text=text, semantics=cropped_images, return_tensors="pt", padding=True).to(device)
inputs.keys(), inputs['input_ids'].shape, inputs['input_semantics'].shape

In [ ]:
inputs['attention_mask']

In [ ]:
attention_mask = torch.ones([2,15])
input_semantics = torch.ones([1,3,224,224])
torch.cat([attention_mask, torch.ones([attention_mask.shape[0], input_semantics.shape[0]])], dim=1).shape

## Dataloader process comprehension

In [ ]:
dataset = load_dataset("cppe-5")

In [ ]:
# Get dataset categories and prepare mappings for label_name <-> label_id
categories = dataset["test"].features["objects"].feature["category"].names
id2label = dict(enumerate(categories))
label2id = {v: k for k, v in id2label.items()}

In [ ]:
validation_transform = A.Compose(
    [A.NoOp()],
    bbox_params=A.BboxParams(format="coco", label_fields=["category"], clip=True),
)

In [ ]:
def to_label_list(id2label):
    return list(id2label.values())

def concat_func(id2label):
    return ". ".join(to_label_list(id2label)) + "."

# Copied from examples/pytorch/object-detection/run_object_detection.format_image_annotations_as_coco
def format_image_annotations_as_coco(
    image_id: str, categories: List[int], areas: List[float], bboxes: List[Tuple[float]]
) -> dict:
    """Format one set of image annotations to the COCO format

    Args:
        image_id (str): image id. e.g. "0001"
        categories (List[int]): list of categories/class labels corresponding to provided bounding boxes
        areas (List[float]): list of corresponding areas to provided bounding boxes
        bboxes (List[Tuple[float]]): list of bounding boxes provided in COCO format
            ([center_x, center_y, width, height] in absolute coordinates)

    Returns:
        dict: {
            "image_id": image id,
            "annotations": list of formatted annotations
        }
    """
    annotations = []
    for category, area, bbox in zip(categories, areas, bboxes):
        formatted_annotation = {
            "image_id": image_id,
            "category_id": category,
            "iscrowd": 0,
            "area": area,
            "bbox": list(bbox),
        }
        annotations.append(formatted_annotation)

    return {
        "image_id": image_id,
        "annotations": annotations,
    }

In [ ]:
import random

def get_random_unique_indices(lst):
    seen = {}
    result = [False] * len(lst)
    
    # Iterate through the list to collect all indices of each unique value
    for i, value in enumerate(lst):
        if value not in seen:
            seen[value] = [i]
        else:
            seen[value].append(i)
    
    # For each unique value, randomly select one index and mark it as True
    for indices in seen.values():
        random_index = random.choice(indices)
        result[random_index] = True
    
    return result


In [ ]:
def group_by_index(values, group_indices):
    # Create a dictionary to hold the grouped values
    grouped_dict = {}
    
    # Iterate through the values and group_indices simultaneously
    for value, group_index in zip(values, group_indices):
        if group_index not in grouped_dict:
            grouped_dict[group_index] = []
        grouped_dict[group_index].append(value)
    
    # Extract the groups in the order of their appearance
    grouped_values = [grouped_dict[key] for key in sorted(grouped_dict.keys())]
    
    return grouped_values


In [ ]:
def augment_and_transform_batch(
    examples: Mapping[str, Any],
    transform: A.Compose,
    processor: AutoProcessor,
    id2label: Dict[int, str],
    label2id: Dict[str, int],
    random_text_prompt: bool = False,
    return_pixel_mask: bool = False,
) -> BatchFeature:
    """
    Apply augmentations and format annotations in COCO format for object detection task.
    Generates the text prompt used. If `random_text_prompt` is False
        then the prompt will follow the same ordering in `id2label` if set to
        True a new ordering will be created and the prompt will be build accordingly
        and labels will be updated as well.

        Example:
            `id2label` -> {'0': 'fish', '1': 'jellyfish', '2': 'penguins', '3':
                        'sharks', '4': 'puffins', '5': 'stingrays', '6': 'starfish'}

            If `random_text_prompt` -> False
                `text` -> "fish. jellyfish. penguins. sharks. puffins. stingrays. starfish."

            If `random_text_prompt` -> True
                `id2label` gets shuffled e.g. {0: 'fish', 1: 'penguins', 2: 'stingrays', 3:
                                            'jellyfish', 4: 'sharks', 5: 'starfish', 6: 'puffins'}
                `text` -> "fish. penguins. stingrays. jellyfish. sharks. starfish. puffins."
    """

    images = []
    annotations = []
    text = []
    semantics = []

    for image_id, image, objects in zip(examples["image_id"], examples["image"], examples["objects"]):
        image = np.array(image.convert("RGB"))

        if random_text_prompt:
            # Original ordering label list
            label_list = to_label_list(id2label)
            # Shuffle label list
            random.shuffle(label_list)
            # Create shuffled id2label
            shuffled_id2label = dict(enumerate(label_list))

            # Mapping of original to shuffled id to update annotations
            old2new = {label2id[label]: new_id for new_id, label in shuffled_id2label.items()}
            prompt = concat_func(shuffled_id2label)
            category = [old2new[category] for category in objects["category"]]
        else:
            prompt = concat_func(id2label)
            category = objects["category"]

        # apply augmentations
        output = transform(image=image, bboxes=objects["bbox"], category=category)
        images.append(output["image"])

        # format annotations in COCO format
        formatted_annotations = format_image_annotations_as_coco(
            image_id, output["category"], objects["area"], output["bboxes"]
        )
        annotations.append(formatted_annotations)
        text.append(prompt)

    # Crop image with aggregated result
    bbox = []
    category = []
    batch_index = []
    for n, annotation in enumerate(annotations):
        for anno in annotation['annotations']:
            bbox.append(anno['bbox'])
            category.append(anno['category_id'])
            batch_index.append(n)

    bool_category = get_random_unique_indices(category)
    bool_category = group_by_index(bool_category, batch_index)
    bbox = group_by_index(bbox, batch_index)

    for image, bool, box in zip(images, bool_category, bbox):
        bboxes = []
        for _bool, _box in zip(bool, box):
            if _bool:
                bboxes.append(_box)
        semantics.extend(crop_bboxes(image, bboxes))
    
    # Apply the image processor transformations: resizing, rescaling, normalization
    result = processor(images=images, text=text, semantics=semantics, images_kwargs={'annotations':annotations}, return_tensors="pt")

    if not return_pixel_mask:
        result.pop("pixel_mask", None)

    return result

In [ ]:
validation_transform_batch = partial(
        augment_and_transform_batch,
        transform=validation_transform,
        processor=processor,
        id2label=id2label,
        label2id=label2id,
        random_text_prompt=True,
    )

In [ ]:
test_dataset = dataset["test"].with_transform(validation_transform_batch)

In [ ]:
test_dataset[0:3].keys()

In [ ]:
model = GroundingDino2ForObjectDetection.from_pretrained("IDEA-Research/grounding-dino-tiny", ignore_mismatched_sizes=True).to(device)

In [ ]:
# model.model.multimodal_backbone.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
return_dict = 'pt'
multimodal_outputs = model.model.multimodal_backbone.vision_model(test_dataset[0:3]['input_semantics'].cuda(), return_dict=return_dict)

In [ ]:
semantic_features = multimodal_outputs.last_hidden_state if return_dict else multimodal_outputs[0]
print(test_dataset[0:3]['input_semantics'].cuda().shape, semantic_features[:, :1, :].shape)

In [ ]:
# Plotting
test_sample = test_dataset[0:3]['input_semantics']
print(test_sample.shape)
fig, axs = plt.subplots(1, test_sample.shape[0], figsize=(8, 8))  # 2x2 grid of subplots

for i in range(test_sample.shape[0]):
    image = transforms.ToPILImage()(test_sample[i])
    axs[i].imshow(image)
    axs[i].set_title('input for clip vision')
    axs[i].axis('off')

plt.tight_layout()  # Adjust subplots to fit into figure area.
plt.show()

In [ ]:
# Plotting
print(semantic_features.shape)
clip_vision_feature = torch.mean(semantic_features[:,1:,:], axis=2).reshape(-1,7,7)
fig, axs = plt.subplots(1, clip_vision_feature.shape[0], figsize=(8, 8))  # 2x2 grid of subplots

for i in range(clip_vision_feature.shape[0]):
    image = transforms.ToPILImage()(clip_vision_feature[i])
    axs[i].imshow(image)
    axs[i].set_title('attention_masks')
    axs[i].axis('off')

plt.tight_layout()  # Adjust subplots to fit into figure area.
plt.show()

## Attention mask visualization

In [ ]:
SPECIAL_TOKENS = [49406, 49407, 269]

def generate_masks_with_special_tokens_and_transfer_map(input_ids: torch.LongTensor) -> Tuple[Tensor, Tensor]:
    """Generate attention mask between each pair of special tokens and positional ids.
    Args:
        input_ids (`torch.LongTensor` of shape `(batch_size, sequence_length)`):
            Indices of input sequence tokens in the vocabulary.
    Returns:
        `tuple(torch.Tensor)` comprising attention mask between each special tokens and position_ids:
        - **attention_mask** (`torch.BoolTensor` of shape `(batch_size, sequence_length)`)
        - **position_ids** (`torch.LongTensor` of shape `(batch_size, sequence_length)`)
    """
    batch_size, num_token = input_ids.shape
    # special_tokens_mask: batch_size, num_token. 1 for special tokens. 0 for normal tokens
    special_tokens_mask = torch.zeros((batch_size, num_token), device=input_ids.device).bool()
    for special_token in SPECIAL_TOKENS:
        special_tokens_mask |= input_ids == special_token

    # idxs: each row is a list of indices of special tokens
    idxs = torch.nonzero(special_tokens_mask)

    # generate attention mask and positional ids
    text_self_attention_masks = torch.eye(num_token, device=input_ids.device).bool().unsqueeze(0).repeat(batch_size, 1, 1)
    position_ids = torch.zeros((batch_size, num_token), device=input_ids.device)
    previous_col = 0
    for i in range(idxs.shape[0]):
        row, col = idxs[i]
        if (col == 0) or (col == num_token - 1):
            text_self_attention_masks[row, col, col] = True
            position_ids[row, col] = 0
        else:
            text_self_attention_masks[row, previous_col + 1 : col + 1, previous_col + 1 : col + 1] = True
            position_ids[row, previous_col + 1 : col + 1] = torch.arange(
                0, col - previous_col, device=input_ids.device
            )

        previous_col = col

    return text_self_attention_masks, position_ids.to(torch.long)

def generate_masks_with_input_semantics(batch_size: int, input_semantics: torch.FloatTensor) -> Tuple[Tensor, Tensor]:
    """Generate attention mask between each pair of special tokens and positional ids.
    Args:
        input_semantics (`torch.FloatTensor` of shape `(semantic_length, 3, width, height)`):
            Indices of input semantics after image processor.
    Returns:
        `tuple(torch.Tensor)` comprising attention mask between each special tokens and position_ids:
        - **attention_mask** (`torch.BoolTensor` of shape `(batch_size, sequence_length)`)
        - **position_ids** (`torch.LongTensor` of shape `(batch_size, sequence_length)`)
    """
    num_semantic, _, width, height = input_semantics.shape

    # generate attention mask and positional ids
    semantic_self_attention_masks = (
        torch.eye(num_semantic, device=input_semantics.device).bool().unsqueeze(0).repeat(batch_size, 1, 1)
    )
    position_ids = torch.zeros((batch_size, num_semantic), device=input_semantics.device)

    return semantic_self_attention_masks, position_ids.to(torch.long)

In [ ]:
text_self_attention_masks, text_position_ids = generate_masks_with_special_tokens_and_transfer_map(inputs['input_ids'])
semantic_self_attention_masks, semantic_position_ids = generate_masks_with_input_semantics(inputs['input_ids'].shape[0], inputs['input_semantics'])

In [ ]:
text_self_attention_masks.shape, text_position_ids.shape, semantic_self_attention_masks.shape, semantic_position_ids.shape

In [ ]:
B, N, _ = text_self_attention_masks.shape
_, M, _ = semantic_self_attention_masks.shape

# Create a new tensor of shape (B, N+M, N+M) filled with zeros
combined = torch.zeros((B, N+M, N+M), dtype=text_self_attention_masks.dtype, device=text_self_attention_masks.device)

# Fill the top-left quadrant with tensor1
combined[:, :N, :N] = text_self_attention_masks

# Fill the bottom-right quadrant with tensor2
combined[:, N:, N:] = semantic_self_attention_masks

In [ ]:
# Plotting
fig, axs = plt.subplots(1, 5, figsize=(8, 8))  # 2x2 grid of subplots

axs[0].imshow(text_self_attention_masks.cpu()[0], cmap='gray')
axs[0].set_title('text_self_attention_masks')
axs[0].axis('off')

axs[1].imshow(text_position_ids.cpu(), cmap='gray')
axs[1].set_title('position_ids')
axs[1].axis('off')

axs[2].imshow(semantic_self_attention_masks.cpu()[0], cmap='gray')
axs[2].set_title('semantic_self_attention_masks')
axs[2].axis('off')

axs[3].imshow(semantic_position_ids.cpu(), cmap='gray')
axs[3].set_title('position_ids')
axs[3].axis('off')

axs[4].imshow(combined.cpu()[0], cmap='gray')
axs[4].set_title('position_ids')
axs[4].axis('off')

plt.tight_layout()  # Adjust subplots to fit into figure area.
plt.show()

## build_label_maps comprehension

In [ ]:
def build_label_maps(logits, input_ids, input_semantics):
    """
    Computes a mapping between the tokens associated with the prompt labels in the logit space with shape `(batch_size, num_labels, hidden_size)`
    where `num_labels` is defined by the number of classes in the input prompt.

    For instance, given the prompt "fish. shark." we get input_ids = [  101,  3869,  1012, 11420,  1012,   102].
    This function will return a mapping for each of the prompt tokens (i.e. tokens associated with "fish" and "shark")
    indicating their position in the logit space.

    """
    hidden_size = logits.shape[-1]
    # Add [PAD] token to the list of special tokens
    delimiter_tokens = torch.tensor(SPECIAL_TOKENS + [0], device=input_ids.device)

    delimiter_token_masks = torch.isin(input_ids, delimiter_tokens)
    print(input_ids)
    print(delimiter_token_masks)
    label_maps = ()
    for delimiter_token_mask in delimiter_token_masks:
        label_map_within_batch = []
        delimiter_indices = torch.where(delimiter_token_mask)[0]
        print(delimiter_indices)
        for i in range(len(delimiter_indices) - 1):
            start = delimiter_indices[i]
            end = delimiter_indices[i + 1]
            if end - start > 1:
                label_map = torch.zeros(hidden_size, device=input_ids.device)
                label_map[start + 1 : end] = 1
                label_map_within_batch.append(label_map)

        label_maps += (torch.stack(label_map_within_batch),)

    return label_maps

In [ ]:
label_maps = build_label_maps(torch.rand(1,900,256), inputs['input_ids'], inputs['input_semantics'])

In [ ]:
for label_map in label_maps:
    print(label_map.shape)
inputs['input_ids'].shape, inputs['input_semantics'].shape, len(label_maps)

In [ ]:
targets = [{'size': tensor([ 755, 1333], device='cuda:0'), 'image_id': tensor([874], device='cuda:0'), 'class_labels': tensor([0, 0, 4, 4, 3, 2, 2, 2, 2], device='cuda:0'), 
'boxes': tensor([[0.2505, 0.5911, 0.3640, 0.8178],
[0.5428, 0.5378, 0.3360, 0.9244],
[0.2829, 0.3114, 0.0495, 0.0788],
[0.5651, 0.2212, 0.0608, 0.0857],
[0.5695, 0.1777, 0.0729, 0.0679],
[0.1891, 0.4508, 0.0571, 0.1260],
[0.3592, 0.4541, 0.0460, 0.1132],
[0.4575, 0.4261, 0.0540, 0.1253],
[0.6351, 0.4460, 0.0548, 0.1124]], device='cuda:0'), 'area': tensor([181113.3750, 205279.8281,   2659.4148,   3987.6768,   3912.0376,
4275.7798,   3212.0142,   4431.3940,   4355.2734], device='cuda:0'), 'iscrowd': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0'), 'orig_size': tensor([1088, 1920], device='cuda:0')}]

In [ ]:
for label_map, target in zip(label_maps, targets):
    print(label_map, target)

## GroundingDino2 Inference

In [ ]:
import requests

import torch
from PIL import Image
from transformers import GroundingDino2ForObjectDetection, GroundingDino2Config, GroundingDino2Processor, GroundingDinoImageProcessor
from transformers import AutoTokenizer, CLIPImageProcessor, AutoProcessor
from typing import Tuple
from torch import Tensor
import matplotlib.pyplot as plt
from datasets import load_dataset
import albumentations as A
from typing import Any, Dict, List, Mapping, Tuple, Union
from transformers.image_processing_utils import BatchFeature
from functools import partial
import numpy as np
import random
import torchvision.transforms as transforms

In [ ]:
print(inputs['input_ids'].shape, inputs['input_semantics'].shape)
input_test_dataset = test_dataset[:1]
for i, j in input_test_dataset.items():
    if type(j) is torch.Tensor:
        input_test_dataset[i] = j.cuda()
    if type(j) is list:
        for k in j:
            for kkk, kk in k.items():
                k[kkk] = kk.cuda()

In [ ]:
output = model(**input_test_dataset)

In [ ]:
tensor = output.logits[0].detach().cpu()
tensor = torch.where(tensor == -float('inf'), torch.tensor(0.0), tensor)
tensor.shape

In [ ]:
plt.plot(tensor[:,:2])

In [ ]:
results = processor.post_process_grounded_object_detection_v2(
    output,
    box_threshold=0.06,
    target_sizes=[raw_image.size[::-1]]
)[0]
print(results)

In [ ]:
import matplotlib.pyplot as plt

# colors for visualization
COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
          [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]

def plot_results(pil_img, scores, probs, boxes):
    plt.figure(figsize=(16,10))
    plt.imshow(pil_img)
    ax = plt.gca()
    colors = COLORS * 100
    for score, prob, (xmin, ymin, xmax, ymax), c in zip(scores, probs, boxes, colors):
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                   fill=False, color=c, linewidth=3))
        label = f'{score:0.2f}'
        ax.text(xmin, ymin, label, fontsize=15,
                bbox=dict(facecolor='yellow', alpha=0.5))
    plt.axis('off')
    plt.show()

In [ ]:
results['prob']

In [ ]:
plot_results(raw_image, results['scores'].detach().cpu(), results['prob'].detach().cpu(), results['boxes'].detach().cpu().numpy())

In [ ]:
torch.cat([torch.rand([3, 15, 256]), torch.rand([3, 5, 256])], dim=1).shape
